# Lesson 2: Data Engineering — Homework

In the real world, documents are broken, have wrong encodings, are empty, or have mismatched extensions. Your task is to learn how to handle them.

**6 tasks.** Each has a `# TODO` — implement the function.

```bash
pip install -r requirements.txt
```

In [ ]:
from pathlib import Path

print("Sample files:")
for f in sorted(Path("samples/enterprise_challenges").iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<40} {size:>10,} bytes")

---
## Task 1: File Encoding Detection

Legacy systems often return files in Windows-1251, Latin-1, or UTF-8 with BOM — without specifying the charset.
Opening such a file with the wrong encoding produces mojibake (garbled text).

**Files to work with:**
- `utf8_with_bom.html` — UTF-8 with BOM marker (extra bytes `\xef\xbb\xbf` at the start)
- `windows1251_no_charset.html` — Windows-1251 without a charset meta tag (Ukrainian legacy systems)
- `latin1_mixed.html` — Latin-1 with German/French characters

**What to do:**
1. Read the file as bytes
2. Detect encoding using `charset_normalizer`
3. Decode the text correctly
4. Strip the BOM if present

In [ ]:
from charset_normalizer import from_bytes


def detect_and_read(file_path: str) -> dict:
    """
    Reads a file, detects its encoding, and returns the decoded text.

    Returns:
    {
        "file": file name,
        "encoding": detected encoding,
        "had_bom": True/False,
        "text": decoded text,
        "char_count": number of characters
    }
    """
    file_path = Path(file_path)
    raw = file_path.read_bytes()

    # Check if the file starts with a BOM (b"\xef\xbb\xbf")
    BOM = b"\xef\xbb\xbf"
    had_bom = raw.startswith(BOM)
    if had_bom:
        raw = raw[len(BOM):]

    # Detect encoding using charset_normalizer
    result = from_bytes(raw).best()
    if result is not None:
        encoding = result.encoding
        text = str(result)
    else:
        # Fallback if detection completely failed
        encoding = "utf-8"
        text = raw.decode("utf-8", errors="replace")

    return {
        "file": file_path.name,
        "encoding": encoding,
        "had_bom": had_bom,
        "text": text,
        "char_count": len(text),
    }

In [ ]:
# Test
encoding_files = [
    "samples/enterprise_challenges/utf8_with_bom.html",
    "samples/enterprise_challenges/windows1251_no_charset.html",
    "samples/enterprise_challenges/latin1_mixed.html",
]

for f in encoding_files:
    result = detect_and_read(f)
    print(f"{result['file']}:")
    print(f"  Encoding: {result['encoding']}")
    print(f"  BOM: {result['had_bom']}")
    print(f"  First 150 chars: {result['text'][:150]}")
    print()

---
## Task 2: Real File Type Detection (magic bytes)

File extensions can lie. Someone saved HTML as `.pdf`, or renamed a binary dump to `.pdf`.
The only reliable way is to check **magic bytes** (the first bytes of the file that identify its format).

**Files to work with:**
- `actually_html.pdf` — HTML content with a .pdf extension
- `actually_pdf.html` — PDF content with a .html extension
- `binary_garbage.pdf` — random bytes with a .pdf extension
- `empty_file.pdf` — empty file (0 bytes)

**What to do:**
1. Determine the file type from its extension (declared type)
2. Determine the real type via the `filetype` library (magic bytes)
3. For HTML files (no magic bytes) — check if content starts with `<html` or `<!doctype`
4. Return whether there is a mismatch

In [ ]:
import filetype as filetype_lib


def detect_file_type(file_path: str) -> dict:
    """
    Determines the real file type and compares it to the extension.

    Returns:
    {
        "file": file name,
        "declared_type": type from extension,
        "detected_type": real type (via magic bytes),
        "is_mismatch": True if types don't match,
        "issue": description of the problem or None
    }
    """
    file_path = Path(file_path)
    declared_type = file_path.suffix.lstrip(".").lower()

    # Check if the file is empty
    if file_path.stat().st_size == 0:
        return {
            "file": file_path.name,
            "declared_type": declared_type,
            "detected_type": None,
            "is_mismatch": False,
            "issue": "empty file",
        }

    # Detect type via filetype (magic bytes)
    kind = filetype_lib.guess(str(file_path))
    if kind is not None:
        detected_type = kind.extension
    else:
        # filetype didn't recognize it — check if it's HTML
        header = file_path.read_bytes()[:512].lower()
        if b"<html" in header or b"<!doctype" in header:
            detected_type = "html"
        else:
            detected_type = None

    # Determine mismatch
    if detected_type is None:
        is_mismatch = False
        issue = None
    elif detected_type != declared_type:
        is_mismatch = True
        issue = f"file is actually {detected_type} but named as .{declared_type}"
    else:
        is_mismatch = False
        issue = None

    return {
        "file": file_path.name,
        "declared_type": declared_type,
        "detected_type": detected_type,
        "is_mismatch": is_mismatch,
        "issue": issue,
    }

In [ ]:
# Test
test_files = [
    "samples/enterprise_challenges/actually_html.pdf",
    "samples/enterprise_challenges/actually_pdf.html",
    "samples/enterprise_challenges/binary_garbage.pdf",
    "samples/enterprise_challenges/empty_file.pdf",
    "samples/enterprise_challenges/normal_report.xlsx",
]

for f in test_files:
    if Path(f).exists():
        result = detect_file_type(f)
        marker = "MISMATCH" if result["is_mismatch"] else "OK"
        print(f"  [{marker}] {result['file']}: declared=.{result['declared_type']}, real={result['detected_type']}")
        if result["issue"]:
            print(f"          -> {result['issue']}")

---
## Task 3: Clean Text Extraction from HTML

Enterprise HTML is a nightmare: 50 levels of nesting from Word exports, navigation bars, sidebars, scripts, cookie banners. Useful content is about 5%.

**Files to work with:**
- `malformed_deeply_nested.html` — 50 levels of `<div>`, unclosed tags, broken entities
- `boilerplate_heavy.html` — 30 nav items, 20 sidebar widgets, 3 paragraphs of useful text

**What to do:**
1. Read HTML using BeautifulSoup
2. Remove `<script>`, `<style>`, `<nav>`, `<header>`, `<footer>` tags
3. Extract clean text
4. Calculate "usefulness" — percentage of text relative to HTML size

In [ ]:
from bs4 import BeautifulSoup


def extract_clean_text(file_path: str) -> dict:
    """
    Extracts clean text from HTML by removing noise.

    Returns:
    {
        "file": file name,
        "raw_size": HTML size in characters,
        "text": clean text,
        "text_size": text size in characters,
        "useful_ratio": percentage of useful text (text_size / raw_size)
    }
    """
    file_path = Path(file_path)
    raw_html = file_path.read_text(errors="replace")

    # Create BeautifulSoup with "html.parser"
    soup = BeautifulSoup(raw_html, "html.parser")

    # Remove all noise tags
    noise_tags = ["script", "style", "nav", "header", "footer", "aside"]
    for tag_name in noise_tags:
        for tag in soup.find_all(tag_name):
            tag.decompose()

    # Extract text
    text = soup.get_text(separator="\n", strip=True)

    raw_size = len(raw_html)
    text_size = len(text)
    useful_ratio = text_size / raw_size if raw_size > 0 else 0

    return {
        "file": file_path.name,
        "raw_size": raw_size,
        "text": text,
        "text_size": text_size,
        "useful_ratio": useful_ratio,
    }

In [ ]:
# Test
html_files = [
    "samples/enterprise_challenges/malformed_deeply_nested.html",
    "samples/enterprise_challenges/boilerplate_heavy.html",
    "samples/enterprise_challenges/multilingual.html",
]

for f in html_files:
    result = extract_clean_text(f)
    print(f"{result['file']}:")
    print(f"  HTML: {result['raw_size']:,} chars -> Text: {result['text_size']:,} chars")
    print(f"  Useful ratio: {result['useful_ratio']:.1%}")
    print(f"  Text: {result['text'][:200]}...")
    print()

---
## Task 4: Handling Broken Files — Safe Parser

Files break: truncated PDFs, corrupted DOCX (broken ZIP), binary garbage with a .pdf extension, empty files, password-protected PDFs.

The parser must never crash — it should return a result or a meaningful error.

**Files to work with:** the entire `enterprise_challenges/` folder — it contains both working and broken files.

**What to do:**
1. Validate the file (empty? correct type?)
2. Try parsing via `unstructured.partition.auto.partition`
3. If error — catch the exception and return a problem description
4. Classify the error: `empty`, `corrupted`, `type_mismatch`, `parse_error`

In [ ]:
from unstructured.partition.auto import partition


def safe_parse(file_path: str) -> dict:
    """
    Safely parses a document — never crashes, always returns a result.

    Returns on success:
    {
        "file": name, "status": "ok",
        "text": extracted text, "char_count": number of characters
    }

    Returns on error:
    {
        "file": name, "status": "error",
        "error_type": error type (empty/corrupted/type_mismatch/parse_error),
        "error_message": description
    }
    """
    file_path = Path(file_path)

    # Check if file is empty (0 bytes)
    if file_path.stat().st_size == 0:
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": "empty",
            "error_message": "File is empty (0 bytes)",
        }

    # Check for file type mismatch
    type_info = detect_file_type(str(file_path))
    if type_info.get("is_mismatch"):
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": "type_mismatch",
            "error_message": type_info.get("issue", "File type mismatch"),
        }

    # Try to parse
    try:
        elements = partition(filename=str(file_path))
        text = "\n".join(str(el) for el in elements)
        return {
            "file": file_path.name,
            "status": "ok",
            "text": text,
            "char_count": len(text),
        }
    except Exception as e:
        err_msg = str(e)
        # Classify the error
        low = err_msg.lower()
        if "corrupt" in low or "invalid" in low or "zip" in low or "eof" in low:
            error_type = "corrupted"
        else:
            error_type = "parse_error"
        return {
            "file": file_path.name,
            "status": "error",
            "error_type": error_type,
            "error_message": err_msg[:200],
        }

In [ ]:
# Test: run safe_parse on the entire enterprise_challenges folder
results = []
challenges_dir = Path("samples/enterprise_challenges")

for f in sorted(challenges_dir.iterdir()):
    if f.is_file():
        result = safe_parse(str(f))
        results.append(result)

ok = [r for r in results if r["status"] == "ok"]
errors = [r for r in results if r["status"] == "error"]

print(f"=== Results: {len(ok)} OK, {len(errors)} ERROR ===\n")

print("OK:")
for r in ok:
    print(f"  {r['file']}: {r['char_count']} chars")

print(f"\nERROR:")
for r in errors:
    print(f"  {r['file']}: [{r['error_type']}] {r['error_message']}")

---
## Task 5: PDF Table Extraction

`unstructured` extracts text from PDFs but **loses table structure** — rows and columns get mixed into plain text. For tables, a specialized tool is needed — `pdfplumber`.

**File to work with:**
- `financial_report_table.pdf` — financial report with two tables (revenue by region, revenue by product)

**What to do:**
1. Extract tables from PDF using `pdfplumber`
2. Convert each table into a list of dicts (first row = keys)
3. Compare the result with what `unstructured` produces — see the difference

In [ ]:
import pdfplumber


def extract_tables_from_pdf(file_path: str) -> list[list[dict]]:
    """
    Extracts tables from a PDF and returns them as a list of tables.
    Each table is a list of dicts (keys = headers from the first row).

    Example:
    [
        [  # table 1
            {"Region": "North America", "Q1": "1,200,000", ...},
            {"Region": "Europe", "Q1": "800,000", ...},
        ],
        [  # table 2
            {"Product": "Enterprise Platform", "Units Sold": "145", ...},
        ],
    ]
    """
    all_tables = []

    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            raw_tables = page.extract_tables()
            for raw_table in raw_tables:
                if not raw_table or len(raw_table) < 2:
                    continue
                headers = raw_table[0]
                rows = []
                for row in raw_table[1:]:
                    row_dict = dict(zip(headers, row))
                    rows.append(row_dict)
                all_tables.append(rows)

    return all_tables

In [ ]:
pdf_path = "samples/enterprise_challenges/financial_report_table.pdf"

# --- pdfplumber: structured tables ---
tables = extract_tables_from_pdf(pdf_path)
print(f"pdfplumber found {len(tables)} tables\n")

for i, table in enumerate(tables):
    print(f"=== Table {i+1}: {len(table)} rows ===")
    if table:
        print(f"  Columns: {list(table[0].keys())}")
        for row in table[:3]:
            print(f"  {row}")
        if len(table) > 3:
            print(f"  ... {len(table) - 3} more rows")
    print()

# --- unstructured: for comparison ---
print("=== unstructured (for comparison) ===")
elements = partition(filename=pdf_path)
text = "\n".join(str(el) for el in elements)
print(text[:500])
print("\n^ See the difference? unstructured loses the table structure.")

---
## Task 6: Chunking a Large Document

For RAG and embeddings, text needs to be split into chunks. With large files (1MB+) there are nuances:
- How does chunk size affect the number of chunks?
- What happens with overlap?
- How long does chunking take?

**File to work with:**
- `huge_audit_log.txt` — ~1.5 MB, 5000 audit log entries

**What to do:**
1. Implement chunking via `RecursiveCharacterTextSplitter`
2. Compare different `chunk_size` values (256, 512, 1024, 2048) — number of chunks, time
3. Compare different `chunk_overlap` values (0, 50, 200) — how overlap affects chunk count

In [ ]:
huge_file = Path("samples/enterprise_challenges/huge_audit_log.txt")
text = huge_file.read_text()
size_mb = huge_file.stat().st_size / 1024 / 1024
print(f"File: {huge_file.name}")
print(f"Size: {size_mb:.2f} MB, {len(text):,} characters")

In [ ]:
import time
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_text(text: str, chunk_size: int = 512, chunk_overlap: int = 50) -> list[str]:
    """
    Splits text into chunks.
    Returns a list of strings (chunks).
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_text(text)
    return chunks

In [ ]:
# Experiment 1: different chunk_size values (overlap=50)
print(f"Text: {len(text):,} characters\n")
print(f"{'chunk_size':>12} | {'chunks':>8} | {'avg length':>10} | {'time (ms)':>10}")
print("-" * 52)

for size in [256, 512, 1024, 2048]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=size, chunk_overlap=50)
    elapsed = (time.time() - t0) * 1000
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{size:>12} | {len(chunks):>8} | {avg:>10.0f} | {elapsed:>10.1f}")

In [ ]:
# Experiment 2: different chunk_overlap values (chunk_size=512)
print(f"chunk_size=512, text: {len(text):,} characters\n")
print(f"{'overlap':>12} | {'chunks':>8} | {'extra':>12} | {'time (ms)':>10}")
print("-" * 52)

baseline = None
for overlap in [0, 50, 100, 200]:
    t0 = time.time()
    chunks = chunk_text(text, chunk_size=512, chunk_overlap=overlap)
    elapsed = (time.time() - t0) * 1000
    if baseline is None:
        baseline = len(chunks)
    extra = len(chunks) - baseline
    print(f"{overlap:>12} | {len(chunks):>8} | +{extra:>10} | {elapsed:>10.1f}")

print(f"\n(overlap increases chunk count because text is duplicated at boundaries)")

In [ ]:
# Look at the boundary between chunks — how overlap works
chunks = chunk_text(text, chunk_size=512, chunk_overlap=100)
print(f"Total chunks: {len(chunks)}\n")

print("=== Chunk 0 (last 150 chars) ===")
print(f"...{chunks[0][-150:]}")
print(f"\n=== Chunk 1 (first 150 chars) ===")
print(f"{chunks[1][:150]}...")
print(f"\n^ See the overlap? The end of chunk 0 repeats at the start of chunk 1.")
print("  This ensures context is not lost at boundaries.")

---
## Done!

You have learned to:
1. **Detect encoding** — charset_normalizer, BOM, legacy encodings
2. **Verify file type** — magic bytes vs extension
3. **Clean HTML** — remove noise, extract useful text
4. **Safely parse** — handle corrupted/empty/broken files without crashing
5. **Extract tables from PDF** — pdfplumber vs unstructured, structured data
6. **Chunking** — how chunk_size and overlap affect results for large documents